# Primes in arithmetic progressions

For fixed $q\geq2$ and $a$, define

$$\pi(x;q,a)=\#\{p\leq x:p\equiv a\pmod q\}.$$

Dirichlet's theorem says that every class with $\gcd(a,q)=1$ contains infinitely many primes. The prime number theorem for arithmetic progressions predicts

$$\pi(x;q,a)\sim\frac{\operatorname{Li}(x)}{\varphi(q)},$$

so the reduced residue classes are asymptotically equidistributed. We examine both this regularity and the finite-range biases it hides.

In [ ]:
from math import gcd, isqrt
from time import perf_counter
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"axes.grid": True, "figure.dpi": 110})
N = 10_000_000

## 1. Generate the primes

This notebook is self-contained and uses an odd-only sieve.

In [ ]:
def odd_sieve(n):
    if n < 2:
        return np.array([], dtype=np.int64)
    odd = np.ones((n+1)//2, dtype=bool); odd[0] = False
    for p in range(3, isqrt(n)+1, 2):
        if odd[p//2]:
            odd[p*p//2::p] = False
    return np.concatenate(([2], 2*np.flatnonzero(odd)+1)).astype(np.int64)

t0 = perf_counter(); primes = odd_sieve(N)
print(f"Generated {len(primes):,} primes through {N:,} in {perf_counter()-t0:.2f} s.")
assert np.searchsorted(primes, 1_000_000, side="right") == 78_498

## 2. Which classes can contain primes?

If $\gcd(a,q)>1$, every number $a+kq$ shares a factor with $q$. Such a class contains at most one prime. We therefore compare only the **reduced residue classes**, whose number is Euler's totient $\varphi(q)$.

In [ ]:
def reduced_classes(q):
    return np.array([a for a in range(q) if gcd(a, q) == 1])

def counts_by_class(prime_data, q, x=None):
    selected = prime_data if x is None else prime_data[:np.searchsorted(prime_data, x, side="right")]
    return np.bincount(selected % q, minlength=q)

for q in [3, 4, 5, 8, 12]:
    classes = reduced_classes(q); counts = counts_by_class(primes, q)
    print(f"q={q:2d}, phi(q)={len(classes):2d}: " +
          ", ".join(f"a={a}: {counts[a]:,}" for a in classes))

## 3. Equidistribution at one cutoff

Compare each reduced class with the equal-share prediction. The finitely many primes dividing $q$ are excluded from the eligible total.

In [ ]:
q = 12  # Try 4, 5, 8, 12, or another small modulus.
classes = reduced_classes(q); counts = counts_by_class(primes, q)
eligible_total = sum(counts[a] for a in classes)
expected = eligible_total/len(classes)
deviation = np.array([counts[a]/expected-1 for a in classes])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(classes.astype(str), [counts[a] for a in classes])
axes[0].axhline(expected, color="black", linestyle="--", label="equal share")
axes[0].set(xlabel="residue a", ylabel="prime count", title=f"Primes modulo {q} through {N:,}"); axes[0].legend()
axes[1].bar(classes.astype(str), 100*deviation); axes[1].axhline(0, color="black", linewidth=1)
axes[1].set(xlabel="residue a", ylabel="percent from equal share", title="Finite-range imbalance")
plt.tight_layout(); plt.show()

## 4. Approach to equal density

The theorem concerns a limit. For every reduced class, the share

$$\frac{\pi(x;q,a)}{\sum_{(b,q)=1}\pi(x;q,b)}$$

should approach $1/\varphi(q)$.

In [ ]:
sample_x = np.unique(np.logspace(2, np.log10(N), 250).astype(np.int64))
positions = np.searchsorted(primes, sample_x, side="right")
residues = primes % q
cumulative = {a: np.cumsum(residues == a) for a in classes}
class_counts = np.vstack([cumulative[a][positions-1] for a in classes])
shares = class_counts/class_counts.sum(axis=0)
plt.figure(figsize=(10, 4.5))
for row, a in zip(shares, classes):
    plt.plot(sample_x, row, label=f"a={a}")
plt.axhline(1/len(classes), color="black", linestyle="--", label=r"$1/\varphi(q)$")
plt.xscale("log"); plt.xlabel("x"); plt.ylabel("share among reduced classes")
plt.title(f"Approach to equidistribution modulo {q}"); plt.legend(ncol=2); plt.show()

## 5. Prime number theorem for progressions

For fixed $q$ and each reduced $a$,

$$\pi(x;q,a)\sim\frac{x}{\varphi(q)\log x}.$$

The normalized ratios below should approach $1$, though not necessarily monotonically.

In [ ]:
model = sample_x/(len(classes)*np.log(sample_x))
plt.figure(figsize=(10, 4.5))
for counts_a, a in zip(class_counts, classes):
    plt.plot(sample_x, counts_a/model, label=f"a={a}")
plt.axhline(1, color="black", linestyle="--"); plt.xscale("log")
plt.xlabel("x"); plt.ylabel(r"$\pi(x;q,a)/(x/(\varphi(q)\log x))$")
plt.title(f"PNT in arithmetic progressions, q={q}"); plt.legend(ncol=2); plt.show()

## 6. Chebyshev's bias modulo 4

Asymptotic equality permits a persistent-looking finite bias. Let

$$D(x)=\pi(x;4,3)-\pi(x;4,1).$$

The data often favor $3\pmod4$. This is **Chebyshev's bias**, but $D(x)$ is not always positive.

In [ ]:
res4 = primes % 4
D_at_primes = np.cumsum(res4 == 3)-np.cumsum(res4 == 1)
D = D_at_primes[positions-1]
plt.figure(figsize=(10, 4.5)); plt.plot(sample_x, D); plt.axhline(0, color="black", linewidth=1)
plt.xscale("log"); plt.xlabel("x"); plt.ylabel(r"$\pi(x;4,3)-\pi(x;4,1)$")
plt.title("Chebyshev's bias modulo 4"); plt.show()
print(f"D({N:,}) = {D_at_primes[-1]:,}")
print(f"Fraction of sampled cutoffs with D(x)>0: {np.mean(D>0):.3f}")

## 7. Local counts

In a window $[x,x+h]$, the heuristic count in one reduced class is $h/(\varphi(q)\log x)$. Global equidistribution can coexist with substantial local fluctuation.

In [ ]:
q_local = 4; classes_local = reduced_classes(q_local)
h = max(10_000, N//100)
starts = np.arange(max(100, N//10), N-h+1, h, dtype=np.int64)
expected_local = h/(len(classes_local)*np.log(starts+h/2))
plt.figure(figsize=(10, 4.5))
for a in classes_local:
    selected = primes[primes % q_local == a]
    local_counts = (np.searchsorted(selected, starts+h, side="right")-
                    np.searchsorted(selected, starts, side="left"))
    plt.plot(starts, local_counts, ".", label=f"a={a}")
plt.plot(starts, expected_local, color="black", linewidth=2, label="heuristic")
plt.xlabel("window start x"); plt.ylabel("primes in window and class")
plt.title(f"Local counts modulo {q_local}, window length {h:,}"); plt.legend(); plt.show()

## Further experiments

1. Repeat for $q=3,5,8,12,20$. Explain why non-reduced classes must be excluded.
2. For every $q\leq30$, measure the largest relative imbalance among its reduced classes.
3. Search all primes, rather than sampled cutoffs, for sign changes of $D(x)$.
4. Compare biases modulo $3$ or $8$; decide which differences to measure first.
5. Change the local window length and quantify the relative fluctuations.
6. Test whether $\sum_{p\leq x,\,p\equiv a\pmod q}1/p$ grows like $(1/\varphi(q))\log\log x$.